# Mitigation Evaluation Analysis
Analyse des résultats de mitigation : toxicity reduction vs content preservation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Chargement des données

In [ ]:
MIT_CSV  = '../../report/mitigation_results.csv'
DET_CSV  = '../../report/detection_predictions.csv'
IMG_ROOT = Path('../../data')  # racine du dataset Kaggle

df = pd.read_csv(MIT_CSV)
print(f'Total rows: {len(df)}')
df.head()

In [ ]:
# Garder uniquement les lignes valides avec prob_after
df = df[df['prob_after'].notna() & (df['error'].isna() | (df['error'] == ''))].copy()

for col in ['prob_before', 'prob_after']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['bertscore_f1', 'clip_score', 'ssim', 'detoxify_before', 'detoxify_after']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# TR% par image
df['tr_pct'] = ((df['prob_before'] - df['prob_after']) / df['prob_before'].replace(0, np.nan)) * 100

hateful = df[df['label_true'] == 1].copy()
print(f'Images valides       : {len(df)}')
print(f'Images hateful (=1)  : {len(hateful)}')
print(f'Images non-hateful (=0): {len(df[df["label_true"]==0])}')

## 2. Axis A — Toxicity Reduction

In [ ]:
pb = hateful['prob_before'].dropna()
pa = hateful['prob_after'].dropna()
tr = hateful['tr_pct'].dropna()

pct_nonhateful = (hateful['prob_after'] < 0.5).mean() * 100

print('AXIS A — Toxicity Reduction')
print(f'  N images hateful mitigées : {len(hateful)}')
print(f'  prob_before moyen         : {pb.mean():.4f}')
print(f'  prob_after  moyen         : {pa.mean():.4f}')
print(f'  TR% moyen                 : {tr.mean():.1f}%')
print(f'  TR% médian                : {tr.median():.1f}%')
print(f'  % images prob_after < 0.5 : {pct_nonhateful:.1f}%')

In [ ]:
# Prob before vs after — scatter + diagonale
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter
ax = axes[0]
ax.scatter(hateful['prob_before'], hateful['prob_after'], alpha=0.5, s=20, color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='pas de changement')
ax.axhline(0.5, color='tomato', linestyle=':', lw=1, label='seuil 0.5')
ax.set_xlabel('prob_before')
ax.set_ylabel('prob_after')
ax.set_title('Probabilité avant vs après mitigation')
ax.legend(fontsize=8)

# Distribution TR%
ax = axes[1]
ax.hist(tr, bins=30, color='steelblue', edgecolor='white')
ax.axvline(tr.mean(), color='tomato', linestyle='--', label=f'moyenne {tr.mean():.1f}%')
ax.axvline(0, color='black', linestyle='-', lw=0.8)
ax.set_xlabel('Toxicity Reduction %')
ax.set_ylabel('Nombre d\'images')
ax.set_title('Distribution du TR%')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Detoxify (si disponible)
det_df = hateful[hateful['detoxify_before'].notna() & hateful['detoxify_after'].notna()]
if len(det_df) > 0:
    print(f'Detoxify (n={len(det_df)})')
    print(f'  Score avant  : {det_df["detoxify_before"].mean():.4f}')
    print(f'  Score après  : {det_df["detoxify_after"].mean():.4f}')
    tr_det = ((det_df['detoxify_before'] - det_df['detoxify_after']) / det_df['detoxify_before'].replace(0, np.nan)) * 100
    print(f'  TR% Detoxify : {tr_det.mean():.1f}%')
else:
    print('Pas de scores Detoxify (relancer --compute_metrics avec detoxify installé)')

## 3. Axis B — Content Preservation

In [ ]:
metrics_b = {
    'BERTScore F1': hateful['bertscore_f1'].dropna(),
    'CLIPScore':    hateful['clip_score'].dropna(),
    'SSIM':         hateful['ssim'].dropna(),
}

print('AXIS B — Content Preservation')
for name, vals in metrics_b.items():
    if len(vals) > 0:
        print(f'  {name:15} n={len(vals):4d}  mean={vals.mean():.4f}  std={vals.std():.4f}')
    else:
        print(f'  {name:15} — pas de données')

In [ ]:
available = {k: v for k, v in metrics_b.items() if len(v) > 0}
if available:
    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, (name, vals) in zip(axes, available.items()):
        ax.hist(vals, bins=25, color='mediumseagreen', edgecolor='white')
        ax.axvline(vals.mean(), color='tomato', linestyle='--', label=f'mean={vals.mean():.3f}')
        ax.set_title(name)
        ax.set_xlabel('Score')
        ax.legend(fontsize=8)
    plt.suptitle('Distribution des métriques de préservation', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Pareto — Trade-off TR% vs BERTScore

In [ ]:
pareto_df = hateful[hateful['bertscore_f1'].notna() & hateful['tr_pct'].notna()].copy()

if len(pareto_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        pareto_df['tr_pct'],
        pareto_df['bertscore_f1'],
        c=pareto_df['prob_before'],
        cmap='RdYlGn_r', alpha=0.6, s=30
    )
    plt.colorbar(sc, ax=ax, label='prob_before')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlabel('Toxicity Reduction % (↑ better)')
    ax.set_ylabel('BERTScore F1 (↑ better)')
    ax.set_title('Pareto : Toxicity Reduction vs Content Preservation')
    plt.tight_layout()
    plt.show()
else:
    print('Pas assez de données pour le Pareto plot (BERTScore manquant)')

## 5. Analyse par type de haine (hate_location)

In [ ]:
loc_df = hateful[hateful['hate_location'].notna() & (hateful['hate_location'] != '')]

if len(loc_df) > 0:
    # Distribution des types
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    loc_df['hate_location'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Distribution des hate_location')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=30)

    # TR% par hate_location
    tr_by_loc = loc_df.groupby('hate_location')['tr_pct'].mean().sort_values()
    tr_by_loc.plot(kind='barh', ax=axes[1], color='mediumseagreen')
    axes[1].set_title('TR% moyen par hate_location')
    axes[1].set_xlabel('TR% moyen')

    plt.tight_layout()
    plt.show()

    # Tableau détaillé
    print('\nDétail par hate_location:')
    loc_df.groupby('hate_location').agg(
        n=('id', 'count'),
        tr_pct_mean=('tr_pct', 'mean'),
        prob_before_mean=('prob_before', 'mean'),
        prob_after_mean=('prob_after', 'mean'),
        bertscore_mean=('bertscore_f1', 'mean'),
    ).round(3)
else:
    print('Pas de données hate_location')

## 6. Métrique jointe — % non-hateful AND cohérent

In [ ]:
CLIP_THRESHOLD = 0.20

joint_df = hateful[hateful['clip_score'].notna()]
if len(joint_df) > 0:
    non_hateful  = joint_df['prob_after'] < 0.5
    coherent     = joint_df['clip_score'] > CLIP_THRESHOLD
    joint        = (non_hateful & coherent).mean() * 100

    print(f'% non-hateful (prob_after < 0.5)               : {non_hateful.mean()*100:.1f}%')
    print(f'% cohérent    (CLIPScore > {CLIP_THRESHOLD})            : {coherent.mean()*100:.1f}%')
    print(f'% non-hateful AND cohérent (joint metric)       : {joint:.1f}%')
else:
    # Sans CLIPScore, juste prob_after
    pct = (hateful['prob_after'] < 0.5).mean() * 100
    print(f'% non-hateful (prob_after < 0.5) : {pct:.1f}%')
    print('CLIPScore non disponible pour la métrique jointe')

## 7. Over-sanitization (label=0)

In [ ]:
nonhat = df[df['label_true'] == 0]
if len(nonhat) > 0:
    triggered = nonhat[nonhat['prob_before'].notna() & (nonhat['prob_before'] >= 0.2)]
    print(f'Images non-hateful (label=0)       : {len(nonhat)}')
    print(f'Wrongly triggered (prob >= 0.2)    : {len(triggered)} ({len(triggered)/max(len(nonhat),1)*100:.1f}%)')
    if len(triggered) > 0:
        triggered[['id', 'prob_before', 'prob_after', 'original_text']].head(10)
else:
    # Charger depuis detection CSV
    try:
        det = pd.read_csv(DET_CSV)
        nonhat_det = det[det['label_true'] == 0]
        nonhat_det['prob_pred'] = pd.to_numeric(nonhat_det['prob_pred'], errors='coerce')
        triggered = nonhat_det[nonhat_det['prob_pred'] >= 0.2]
        print(f'Images non-hateful (label=0)       : {len(nonhat_det)}')
        print(f'Wrongly triggered (prob >= 0.2)    : {len(triggered)} ({len(triggered)/max(len(nonhat_det),1)*100:.1f}%)')
    except:
        print('Charger detection_predictions.csv pour cette analyse')

## 8. Visualisation avant/après (exemples)

In [ ]:
def show_before_after(rows, img_root, n=4):
    rows = rows.head(n)
    fig, axes = plt.subplots(len(rows), 2, figsize=(10, 4 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for ax_row, (_, r) in zip(axes, rows.iterrows()):
        orig_path = img_root / r['img']
        mit_path  = Path(r['mitigated_path']) if pd.notna(r.get('mitigated_path')) else None
        for ax, (path, title) in zip(ax_row, [
            (orig_path, f"Original\nprob={r['prob_before']:.2f}"),
            (mit_path,  f"Mitigated\nprob={r['prob_after']:.2f}  TR={r['tr_pct']:.0f}%")
        ]):
            if path and Path(path).exists():
                ax.imshow(Image.open(path).convert('RGB'))
            else:
                ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
                ax.set_facecolor('#eee')
            ax.set_title(title, fontsize=9)
            ax.axis('off')
    plt.suptitle('Avant / Après mitigation', y=1.01)
    plt.tight_layout()
    plt.show()

# Exemples avec le TR% le plus élevé
best_mit = hateful[hateful['mitigated_path'].notna()].nlargest(4, 'tr_pct')
show_before_after(best_mit, IMG_ROOT)

In [ ]:
# Exemples où la mitigation a échoué (TR% négatif = prob a augmenté)
worst_mit = hateful[hateful['mitigated_path'].notna()].nsmallest(4, 'tr_pct')
print('Cas où la mitigation a le moins bien fonctionné :')
show_before_after(worst_mit, IMG_ROOT)

In [ ]:
# MPS par hate_location
loc_mps = mps_rows[mps_rows['hate_location'].notna() & (mps_rows['hate_location'] != '')]
if len(loc_mps) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    loc_mps.boxplot(column='mps', by='hate_location', ax=ax)
    ax.set_title('MPS par type de haine')
    ax.set_xlabel('hate_location')
    ax.set_ylabel('MPS')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

    print('MPS moyen par hate_location:')
    display(loc_mps.groupby('hate_location')['mps'].agg(['mean', 'std', 'count']).round(4))

# Meilleur vs pire MPS
for label, subset in [
    ('Meilleur MPS — sens bien préservé', mps_rows.nlargest(5, 'mps')),
    ('Pire MPS — sens perdu',             mps_rows.nsmallest(5, 'mps')),
]:
    print(f'\n{label}')
    display(subset[['id', 'mps', 'tr_pct', 'hate_location', 'original_text', 'replacement_text']].reset_index(drop=True))

In [ ]:
print('=' * 50)
print('RÉSUMÉ MITIGATION')
print('=' * 50)
print(f'  N images mitigées       : {len(hateful)}')
print()
print('  AXIS A — Toxicity Reduction')
print(f'    prob_before moyen     : {hateful["prob_before"].mean():.4f}')
print(f'    prob_after  moyen     : {hateful["prob_after"].mean():.4f}')
print(f'    TR% moyen             : {hateful["tr_pct"].mean():.1f}%')
print(f'    % prob_after < 0.5    : {(hateful["prob_after"]<0.5).mean()*100:.1f}%')
print()
print('  AXIS B — Content Preservation')
for name, col in [('BERTScore F1', 'bertscore_f1'), ('CLIPScore', 'clip_score'), ('SSIM', 'ssim')]:
    vals = hateful[col].dropna()
    if len(vals) > 0:
        print(f'    {name:15}: {vals.mean():.4f}')
    else:
        print(f'    {name:15}: —')
if 'mps' in mps_rows.columns:
    print(f'    {"MPS":15}: {valid_mps.mean():.4f}')
print('=' * 50)

In [ ]:
mps_rows = hateful[
    hateful['mitigated_path'].notna() &
    hateful['original_text'].notna()
].copy()

mps_scores = []
for _, r in mps_rows.iterrows():
    try:
        orig_img = Image.open(IMG_ROOT / r['img']).convert('RGB')
        mit_img  = Image.open(r['mitigated_path']).convert('RGB')
        orig_txt = str(r['original_text']).replace('\\n', ' ')
        repl_txt = str(r.get('replacement_text') or orig_txt).replace('\\n', ' ')

        emb_orig = multimodal_embedding(clip_model, clip_proc, orig_img, orig_txt)
        emb_mit  = multimodal_embedding(clip_model, clip_proc, mit_img,  repl_txt)
        mps_scores.append((emb_orig * emb_mit).sum().item())
    except Exception as e:
        mps_scores.append(float('nan'))

mps_rows = mps_rows.copy()
mps_rows['mps'] = mps_scores
valid_mps = mps_rows['mps'].dropna()

print(f'MPS moyen  : {valid_mps.mean():.4f}')
print(f'MPS médian : {valid_mps.median():.4f}')
print(f'MPS std    : {valid_mps.std():.4f}')

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

def multimodal_embedding(clip_model, clip_proc, image, text):
    inputs = clip_proc(
        text=[text or ""], images=[image],
        return_tensors="pt", padding=True, truncation=True, max_length=77
    )
    with torch.no_grad():
        out = clip_model(**inputs)
    img_e = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt_e = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
    return (img_e + txt_e) / 2

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP loaded")

## 9. MPS — Multimodal Preservation Score

Mesure à quel point le **sens global** du mème (image + texte ensemble) est préservé après mitigation.

```
MPS = cosine_similarity(
    CLIP(image_originale + texte_original),
    CLIP(image_mitigée  + texte_remplacé)
)
```

Proche de 1 = le mème mitigé "veut dire la même chose" que l'original (sans la haine).

## 9. Résumé

In [ ]:
print('=' * 50)
print('RÉSUMÉ MITIGATION')
print('=' * 50)
print(f'  N images mitigées       : {len(hateful)}')
print()
print('  AXIS A — Toxicity Reduction')
print(f'    prob_before moyen     : {hateful["prob_before"].mean():.4f}')
print(f'    prob_after  moyen     : {hateful["prob_after"].mean():.4f}')
print(f'    TR% moyen             : {hateful["tr_pct"].mean():.1f}%')
print(f'    % prob_after < 0.5    : {(hateful["prob_after"]<0.5).mean()*100:.1f}%')
print()
print('  AXIS B — Content Preservation')
for name, col in [('BERTScore F1', 'bertscore_f1'), ('CLIPScore', 'clip_score'), ('SSIM', 'ssim')]:
    vals = hateful[col].dropna()
    if len(vals) > 0:
        print(f'    {name:15}: {vals.mean():.4f}')
    else:
        print(f'    {name:15}: —')
print('=' * 50)